In [1]:
from google.colab import drive
# 运行此单元格后，点击弹出的链接授权访问您的 Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split

dataset = pd.read_csv('/content/drive/MyDrive/disaster-tweets-prediction/data/train.csv')

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\W+', ' ', text)

    return text

dataset['text'] = dataset['text'].apply(clean_text)
text = dataset["text"].tolist()
target = dataset["target"].tolist()
x_train, x_eval, y_train, y_eval = train_test_split(text, target, test_size=0.2)

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

tfidf_vect = TfidfVectorizer(ngram_range=(1, 2))
X_train_tfidf = tfidf_vect.fit_transform(x_train)
y_train = np.array(y_train)

X_train_tfidf.shape

(6090, 65791)

In [4]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression().fit(X_train_tfidf, y_train)

X_eval_tfidf = tfidf_vect.transform(x_eval)
y_eval = np.array(y_eval)
eval_predicted = clf.predict(X_eval_tfidf)

from sklearn import metrics

print(metrics.classification_report(y_eval, eval_predicted))
metrics.confusion_matrix(y_eval, eval_predicted)

              precision    recall  f1-score   support

           0       0.81      0.90      0.85       879
           1       0.84      0.71      0.77       644

    accuracy                           0.82      1523
   macro avg       0.82      0.80      0.81      1523
weighted avg       0.82      0.82      0.82      1523



array([[792,  87],
       [189, 455]])

In [5]:
test = pd.read_csv('/content/drive/MyDrive/disaster-tweets-prediction/data/test.csv')

test['text'] = test['text'].apply(clean_text)
test_text = test["text"].tolist()

X_test_tfidf = tfidf_vect.transform(test_text)
test_predicted = clf.predict(X_test_tfidf)

df = pd.DataFrame(test_predicted, columns=['target'])
submission_logic = pd.DataFrame({'id': test['id'], 'target': test_predicted})

submission_logic.to_csv('/content/drive/MyDrive/disaster-tweets-prediction/outputs/submission_logistic.csv', index=False)
print("Submission saved to Google Drive!")

Submission saved to Google Drive!


## Optimization solution: Use LinearSVC and improved TF-IDF

In [8]:
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline

# Establish a pipeline that integrates feature extraction and classifiers.
# Using sublinear_tf=True can smooth out the influence of extremely high-frequency words.
# LinearSVC typically performs better when handling high-dimensional sparse features (such as TF-IDF).
text_clf = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2),
                             max_features=50000,
                             sublinear_tf=True)),
    ('clf', LinearSVC(C=0.5, random_state=42))
])

text_clf.fit(x_train, y_train)

optimized_predicted = text_clf.predict(x_eval)
print("Optimized classification report：")
print(metrics.classification_report(y_eval, optimized_predicted))
print("Confusion Matrix：")
display(metrics.confusion_matrix(y_eval, optimized_predicted))

Optimized classification report：
              precision    recall  f1-score   support

           0       0.81      0.86      0.84       879
           1       0.79      0.73      0.76       644

    accuracy                           0.80      1523
   macro avg       0.80      0.80      0.80      1523
weighted avg       0.80      0.80      0.80      1523

Confusion Matrix：


array([[755, 124],
       [173, 471]])

In [9]:
optimized_test_predicted = text_clf.predict(test_text)
submission_optimized = pd.DataFrame({'id': test['id'], 'target': optimized_test_predicted})

output_path = '/content/drive/MyDrive/disaster-tweets-prediction/outputs/submission_linearSVC.csv'
submission_optimized.to_csv(output_path, index=False)
print(f"Submission_linearSVC saved to Google Drive!")

Submission_linearSVC saved to Google Drive!
